In [1]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        gpt_input_dim = num_features
        gpt_hidden_dim_internal = num_features # 为了匹配原始 train.py 的 hidden_dim=factors
        gpt_num_heads_internal = 4
        gpt_ff_expansion_factor_internal = 128 
        model = GPT(
            input_dim=gpt_input_dim,                   # 传递原始输入维度
            hidden_dim=gpt_hidden_dim_internal,        # 模型内部工作维度
            num_heads=gpt_num_heads_internal,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_ff_expansion_factor_internal, # 使用原始值
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 可以使用默认值或从命令行获取
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")



import sys # 确保导入sys模块

# ... (你的 main_deep_learning_trainer.py 脚本的其他导入和函数定义应该在这里) ...

if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 Transformer 在 8因子 MIMIC-III + eICU 数据上 ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_eICU_8_factors_split', # 8因子预分割数据目录
        '--model_name', 'Transformer',                                       # 模型名称
        # --- Transformer 特定参数 ---
        '--model_dim', '128',       # Transformer 内部维度
        '--depth', '8',             # Transformer 深度 (block数量)
        '--num_heads', '8',         # Transformer 注意力头数
        '--drop_ratio', '0.1',      # Dropout 比率 (与原始摘要一致)
        # '--drop_path_ratio', '0.1', # 如果你的Transformer类使用它
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.001',            # 学习率 (与原始摘要一致)
        '--epochs', '1400',         # Epochs (与原始摘要一致)
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 输出基础目录
        '--experiment_tag', 'mimic3eICU-8f-transformer-replication', # 实验标签
        '--seed', '42',             # 随机种子
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead',   # 标签列名
        # '--num_layers', '2',      # 对于Transformer，depth参数更相关
    ]
    # --------------------------------------------------------------------

    # 调用你的主函数
    main()



--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 1400
  experiment_tag: mimic3eICU-8f-transformer-replication
  label_column: dead
  lr: 0.001
  model_dim: 128
  model_name: Transformer
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_eICU_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/Transformer_mimic3_eICU_8_factors_mimic3eICU-8f-transformer-replication_trainseed42_20250610-041619
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 8775
  - 测试样本数: 3762
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split，特征数: 8

--- 模型结构 ---
Transformer(
  (emmb): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): LayerNorm((128,), e

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.4532 at Epoch 1. Model and DeLong data updated.
Epoch [001/1400] | Train Loss: 416.8643 | Test Acc: 0.4979 | Test AUC: 0.4532 | Time: 0.75s
  * New best Test AUC: 0.5746 at Epoch 4. Model and DeLong data updated.
Epoch [004/1400] | Train Loss: 371.6958 | Test Acc: 0.5021 | Test AUC: 0.5746 | Time: 0.25s
  * New best Test AUC: 0.5751 at Epoch 5. Model and DeLong data updated.
Epoch [005/1400] | Train Loss: 362.7633 | Test Acc: 0.4979 | Test AUC: 0.5751 | Time: 0.40s
  * New best Test AUC: 0.5752 at Epoch 30. Model and DeLong data updated.
Epoch [030/1400] | Train Loss: 353.9281 | Test Acc: 0.5502 | Test AUC: 0.5752 | Time: 0.25s
  * New best Test AUC: 0.5752 at Epoch 31. Model and DeLong data updated.
Epoch [031/1400] | Train Loss: 354.2246 | Test Acc: 0.5524 | Test AUC: 0.5752 | Time: 0.25s
  * New best Test AUC: 0.5754 at Epoch 32. Model and DeLong data updated.
Epoch [032/1400] | Train Loss: 353.4700 | Test Acc: 0.5529 | Test AUC: 0.5754 | Time: 0.38s
  * New

In [2]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        gpt_input_dim = num_features
        gpt_hidden_dim_internal = num_features # 为了匹配原始 train.py 的 hidden_dim=factors
        gpt_num_heads_internal = 4
        gpt_ff_expansion_factor_internal = 128 
        model = GPT(
            input_dim=gpt_input_dim,                   # 传递原始输入维度
            hidden_dim=gpt_hidden_dim_internal,        # 模型内部工作维度
            num_heads=gpt_num_heads_internal,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_ff_expansion_factor_internal, # 使用原始值
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 可以使用默认值或从命令行获取
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")




if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 Lstm 在 8因子 MIMIC-III + eICU 数据上 ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_eICU_8_factors_split', # 8因子预分割数据目录
        '--model_name', 'Lstm',                                              # 模型名称
        # --- Lstm 相关参数 ---
        '--drop_ratio', '0.1',      # Dropout 比率 (Lstm类使用)
        '--num_layers', '2',        # 命令行参数，但你的Lstm实现可能不使用它
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.001',            # 学习率 (与原始摘要一致)
        '--epochs', '2000',         # Epochs (与原始摘要一致)
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 输出基础目录
        '--experiment_tag', 'mimic3eICU-8f-lstm-replication',   # 实验标签
        '--seed', '42',             # 随机种子
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead'    # 标签列名
    ]
    main()



--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 2000
  experiment_tag: mimic3eICU-8f-lstm-replication
  label_column: dead
  lr: 0.001
  model_dim: 128
  model_name: Lstm
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_eICU_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/Lstm_mimic3_eICU_8_factors_mimic3eICU-8f-lstm-replication_trainseed42_20250610-043211
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 8775
  - 测试样本数: 3762
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split，特征数: 8
初始化 Lstm模型: factors=8, batch_size=5000, drop_ratio=0.1

--- 模型结构 ---
Lstm(
  (backbone1): LSTM(8, 192)
  (backbone2): LSTM(192, 192)
  (backbone3): LSTM(192, 32)
  (

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.5300 at Epoch 1. Model and DeLong data updated.
Epoch [001/2000] | Train Loss: 378.9810 | Test Acc: 0.5021 | Test AUC: 0.5300 | Time: 0.45s
  * New best Test AUC: 0.5397 at Epoch 2. Model and DeLong data updated.
Epoch [002/2000] | Train Loss: 372.8617 | Test Acc: 0.5021 | Test AUC: 0.5397 | Time: 0.42s
  * New best Test AUC: 0.5429 at Epoch 4. Model and DeLong data updated.
Epoch [004/2000] | Train Loss: 362.1519 | Test Acc: 0.5027 | Test AUC: 0.5429 | Time: 0.31s
  * New best Test AUC: 0.5463 at Epoch 5. Model and DeLong data updated.
Epoch [005/2000] | Train Loss: 358.8851 | Test Acc: 0.5412 | Test AUC: 0.5463 | Time: 0.27s
  * New best Test AUC: 0.5560 at Epoch 6. Model and DeLong data updated.
Epoch [006/2000] | Train Loss: 358.1705 | Test Acc: 0.5388 | Test AUC: 0.5560 | Time: 0.41s
  * New best Test AUC: 0.5692 at Epoch 8. Model and DeLong data updated.
Epoch [008/2000] | Train Loss: 356.8763 | Test Acc: 0.5470 | Test AUC: 0.5692 | Time: 0.27s
  * New be

In [3]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        gpt_input_dim = num_features
        gpt_hidden_dim_internal = num_features # 为了匹配原始 train.py 的 hidden_dim=factors
        gpt_num_heads_internal = 4
        gpt_ff_expansion_factor_internal = 128 
        model = GPT(
            input_dim=gpt_input_dim,                   # 传递原始输入维度
            hidden_dim=gpt_hidden_dim_internal,        # 模型内部工作维度
            num_heads=gpt_num_heads_internal,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_ff_expansion_factor_internal, # 使用原始值
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 可以使用默认值或从命令行获取
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")





if __name__ == '__main__':

    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_eICU_8_factors_split', # 8因子预分割数据目录
        '--model_name', 'GRU',                                               # 模型名称
        # --- GRU 特定参数 ---
        '--num_layers', '2',        # GRU 的层数 (与原始摘要一致)
        '--drop_ratio', '0.1',      # Dropout 比率 (与原始摘要一致)
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.0005',           # 学习率 (与原始摘要一致)
        '--epochs', '3000',         # Epochs (与原始摘要一致)
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 输出基础目录
        '--experiment_tag', 'mimic3eICU-8f-gru-replication',    # 实验标签
        '--seed', '42',             # 随机种子
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead'    # 标签列名
    ]
    # --------------------------------------------------------------------

    # 调用你的主函数
    main()



--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 3000
  experiment_tag: mimic3eICU-8f-gru-replication
  label_column: dead
  lr: 0.0005
  model_dim: 128
  model_name: GRU
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_eICU_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/GRU_mimic3_eICU_8_factors_mimic3eICU-8f-gru-replication_trainseed42_20250610-050443
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 8775
  - 测试样本数: 3762
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split，特征数: 8

--- 模型结构 ---
GRU(
  (gru): GRU(8, 128, num_layers=2)
  (head1): Sequential(
    (0): Linear(in_features=8, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropou

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.5667 at Epoch 1. Model and DeLong data updated.
Epoch [001/3000] | Train Loss: 365.6733 | Test Acc: 0.4979 | Test AUC: 0.5667 | Time: 0.27s
  * New best Test AUC: 0.5683 at Epoch 2. Model and DeLong data updated.
Epoch [002/3000] | Train Loss: 356.0936 | Test Acc: 0.4979 | Test AUC: 0.5683 | Time: 0.21s
  * New best Test AUC: 0.5710 at Epoch 3. Model and DeLong data updated.
Epoch [003/3000] | Train Loss: 355.7664 | Test Acc: 0.4979 | Test AUC: 0.5710 | Time: 0.21s
  * New best Test AUC: 0.5746 at Epoch 4. Model and DeLong data updated.
Epoch [004/3000] | Train Loss: 354.1292 | Test Acc: 0.4979 | Test AUC: 0.5746 | Time: 0.36s
  * New best Test AUC: 0.5767 at Epoch 5. Model and DeLong data updated.
Epoch [005/3000] | Train Loss: 353.0849 | Test Acc: 0.4979 | Test AUC: 0.5767 | Time: 0.21s
  * New best Test AUC: 0.5786 at Epoch 6. Model and DeLong data updated.
Epoch [006/3000] | Train Loss: 353.3175 | Test Acc: 0.4979 | Test AUC: 0.5786 | Time: 0.21s
  * New be

In [4]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        gpt_input_dim = num_features
        gpt_hidden_dim_internal = num_features # 为了匹配原始 train.py 的 hidden_dim=factors
        gpt_num_heads_internal = 4
        gpt_ff_expansion_factor_internal = 128 
        model = GPT(
            input_dim=gpt_input_dim,                   # 传递原始输入维度
            hidden_dim=gpt_hidden_dim_internal,        # 模型内部工作维度
            num_heads=gpt_num_heads_internal,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_ff_expansion_factor_internal, # 使用原始值
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 可以使用默认值或从命令行获取
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")



if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 GPT 在 8因子 MIMIC-III + eICU 数据上 ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_eICU_8_factors_split', # 8因子预分割数据目录
        '--model_name', 'GPT',                                               # 模型名称
        # --- GPT 特定参数 (基于原始 train.py 中的实例化逻辑和摘要信息) ---
        # hidden_dim 将在 initialize_model 中设为 num_features (8)
        # num_heads 将在 initialize_model 中硬编码为 4
        # ff_expansion_factor 将在 initialize_model 中硬编码为 128
        '--num_layers', '1',        # GPT 的层数 (与原始摘要一致)
        '--drop_ratio', '0.1',      # Dropout 比率 (与原始摘要一致)
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.005',            # 学习率 (与原始摘要一致)
        '--epochs', '10000',        # Epochs (与原始摘要一致，极高!)
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 输出基础目录
        '--experiment_tag', 'mimic3eICU-8f-gpt-replication',    # 实验标签
        '--seed', '42',             # 随机种子
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead',   # 标签列名
        # --- 以下参数如果命令行中提供了，initialize_model for GPT 会有特殊处理以复现原始行为 ---
        # '--model_dim', '128', # GPT的hidden_dim会用num_features (8)
        # '--num_heads', '8',   # GPT的num_heads会硬编码为4
    ]
    # 调用你的主函数
    main()



--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 10000
  experiment_tag: mimic3eICU-8f-gpt-replication
  label_column: dead
  lr: 0.005
  model_dim: 128
  model_name: GPT
  num_heads: 8
  num_layers: 1
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_eICU_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/GPT_mimic3_eICU_8_factors_mimic3eICU-8f-gpt-replication_trainseed42_20250610-051542
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 8775
  - 测试样本数: 3762
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_eICU_8_factors_split，特征数: 8

--- 模型结构 ---
GPT(
  (input_projection): Linear(in_features=8, out_features=8, bias=True)
  (pos_embedding): Embedding(512, 8)
  (dropout): Dropout(p=0.1, inplace=False

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.4687 at Epoch 1. Model and DeLong data updated.
Epoch [001/10000] | Train Loss: 695.2426 | Test Acc: 0.4979 | Test AUC: 0.4687 | Time: 0.85s
  * New best Test AUC: 0.4850 at Epoch 2. Model and DeLong data updated.
Epoch [002/10000] | Train Loss: 476.0922 | Test Acc: 0.4979 | Test AUC: 0.4850 | Time: 0.32s
  * New best Test AUC: 0.4884 at Epoch 204. Model and DeLong data updated.
Epoch [204/10000] | Train Loss: 343.3514 | Test Acc: 0.4979 | Test AUC: 0.4884 | Time: 0.20s
  * New best Test AUC: 0.4909 at Epoch 206. Model and DeLong data updated.
Epoch [206/10000] | Train Loss: 339.6937 | Test Acc: 0.4979 | Test AUC: 0.4909 | Time: 0.20s
  * New best Test AUC: 0.4958 at Epoch 208. Model and DeLong data updated.
Epoch [208/10000] | Train Loss: 340.3017 | Test Acc: 0.4979 | Test AUC: 0.4958 | Time: 0.20s
  * New best Test AUC: 0.5020 at Epoch 210. Model and DeLong data updated.
Epoch [210/10000] | Train Loss: 341.0722 | Test Acc: 0.4979 | Test AUC: 0.5020 | Time: 0.